In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm
from astropy.io import fits
from datetime import datetime
import glob
from show import *
from utils import *
from view import *
from reprojection import *
from make_map import *


In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])


#t = np.where(np.all([dates > datetime(2026,7,15),
#                     dates < datetime(2026,9,6)], axis=0))[0] # CR2314

#t = np.where(np.all([dates > datetime(2026,8,20),
#                     dates < datetime(2026,10,6)], axis=0))[0] # CR2315

t = np.where(np.all([dates > datetime(2026,7,15),
                     dates < datetime(2026,10,6)], axis=0))[0] # CR2314 + CR2315


files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260715T030009_V202610200227C_0647150501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20261005T093003_V202610191635C_0650050504.fits'),
 268)

In [3]:
data, view = make_map(files[:],
                      correct_dr=True,
                      kind='synoptic',
                      correct_bias=True,
                      #car_rot=2
                      )

/home/ulyanov/PycharmProjects/reprojection/src/make_map.py:54: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))
/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:19: RuntimeWarning: divide by zero encountered in divide
  return data / self.mu


In [29]:
show_data(data, view, vmin=-500, vmax=500, cmap='hmimag')

/tmp/ipykernel_10191/177953628.py:138: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


(<Figure size 1500x600 with 2 Axes>,
 <Axes: xlabel='Carrington Longitude', ylabel='Carrington Latitude'>)

In [5]:
s = np.load('/home/ulyanov/data/sdo/hmi/sft1d.npz')
flux_hmi = (s['fluxes_raw'][-1] + s['fluxes_raw'][-2]) / 2
flux_hmi_ = (s['fluxes_filled'][-1] + s['fluxes_filled'][-2]) / 2
flux_sft = (s['fluxes_sft'][-1] + s['fluxes_sft'][-2]) / 2
lat_hmi = s['lat']

In [6]:
nx = len(data)
#lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
flux = np.nanmean(data, axis=1)
#flux[lat < -65] = np.nan


plt.figure(figsize=(10,8))

plt.plot(lat, flux, lw=2, label='FDT', color='blue')
plt.plot(lat_hmi, flux_hmi, lw=1.5, label='HMI', color='red', alpha=0.7)
plt.plot(lat_hmi, flux_sft, '--', lw=1.5, label='SFT', color='black', alpha=0.7)
#plt.plot(lat_hmi, flux_hmi_, lw=1.5, label='HMI', color='red', alpha=0.7)

#plt.title('2314')
plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-8,8)
plt.grid(True)
plt.legend()
plt.tight_layout()

In [7]:
np.savez('CR2315_synoptic.npz', data=data, lat=lat)

In [9]:
s = np.load('CR2315_synoptic.npz')
data = s['data']
lat = s['lat']

In [13]:
np.mean(data) / np.mean(np.abs(data))

np.float64(-0.013962828292082162)

In [11]:
s = np.load('/home/ulyanov/data/sdo/hmi/sft1d.npz')
fluxes_raw = s['fluxes_raw']
fluxes_sft = s['fluxes_sft']
lat = s['lat']

In [12]:
t = (np.abs(lat) > 50) & (np.abs(lat) < 60)
np.sqrt(np.nanstd(fluxes_raw[:,t] - fluxes_sft[:,t]) ** 2 - np.std(np.gradient(fluxes_raw, axis=1)[:,t]) ** 2 * 2)

np.float64(0.443946093775851)